In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

TABLE = "data/historic_airphoto_index/main_index_blank.gpkg"
OUTPUT_DIR = Path("data/historic_airphoto_index/subsets")

gdf = gpd.read_file(TABLE)

## Prepare the Subset

In [ ]:
# CONFIGURATION CELL - EDIT HERE

# specify the county
COUNTY = "Carver"

### Examine available years for a county

In [ ]:
mask = gdf["county"] == COUNTY

subset = gdf.loc[mask]

for year in sorted(subset["year"].dropna().unique()):
    print(year)

### Choose Years and Manually Populate Scale Info
Use scale information from PDFs found here: 
https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery

This notebook assumes you're going to choose a set of years that all have the same scale, for efficiency. 

In [ ]:
# CONFIGURATION CELL - EDIT HERE

# Choose years for processing
# These should all have the same scale
YEARS = [1937, 1938, 1940, 1951, 1957, 1963]

SCALE = 20000

In [ ]:
mask = (
    (gdf["county"] == COUNTY) &
    (gdf["year"].isin(YEARS))
)

subset_gdf = gdf.loc[mask].copy()

print(f"Matching records: {len(subset_gdf):,}")

In [ ]:
missing_scale = subset_gdf["scale"].isna()

subset_gdf.loc[missing_scale, "scale"] = SCALE

print(f"Scales populated: {missing_scale.sum():,}")
print(subset_gdf["scale"].value_counts(dropna=False))

In [ ]:
# check for unexpected scale values
assert subset_gdf["scale"].eq(SCALE).all(), (
    "Some records have a different scale than expected!"
)

In [ ]:
subset_gdf

### Save a subset index table with a fancy name

In [ ]:
# Build filename components
county_name = COUNTY.strip().lower().replace(" ", "_")

year_suffix = "_".join(
    f"{year % 100:02d}" for year in sorted(set(YEARS))
)

filename = f"subset_{SCALE}_{county_name}_{year_suffix}.gpkg"

# Create output directory if needed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / filename

# Prevent accidental overwrites
if output_path.exists():
    raise FileExistsError(f"File already exists: {output_path}")

# Save subset
subset_gdf.to_file(output_path, layer="photos", driver="GPKG")

print(f"Saved {len(subset_gdf):,} records to:")
print(output_path)

In [ ]:
subset_gdf.columns

### Optional editing cells

In [ ]:
# edit subset

edited_gdf = gpd.read_file("data/historic_airphoto_index/subsets/subset_20000_carver_37_38_40_51_57_63.gpkg")

In [ ]:
edited_gdf

In [ ]:
edited_gdf["error"] = None

In [ ]:
edited_gdf = edited_gdf.rename(columns={
    "date_download" : "date_prepared",
})

In [ ]:
edited_gdf = edited_gdf[['cty_year_id','status','county','year','photo_id','h_extent','ground_width','gcp_count','confidence','scale','dpi_x','width_px','worldfile_json','url','flight_code','source_id','date_prepared','date_georef','error','latitude','geometry']]

In [ ]:
edited_gdf

In [ ]:
edited_gdf.to_file("data/historic_airphoto_index/subsets/subset_20000_carver_37_38_40_51_57_63.gpkg", layer="photos", driver="GPKG")